In [ ]:
from data import stratified_split, WignerDataset
from pathlib import Path
import numpy as np

# clean
rng = np.random.default_rng(67)
trc,tec,vac = stratified_split(3500, rng, 80, 10, 10)

# noisy
rng = np.random.default_rng(64)
trn,ten,van = stratified_split(3500, rng, 80, 10, 10)

# tu daj path do danych
project_dir = Path.cwd().parent.parent
x = project_dir / f"data/train_noisy_3500.h5"
y = project_dir / f"data/train_clean_3500.h5"

train_ds_noisy = WignerDataset(x, indices=trn)
test_ds_noisy = WignerDataset(x, indices=ten)
val_ds_noisy = WignerDataset(x, indices=van)

train_ds_clean = WignerDataset(x, indices=trc)
test_ds_clean = WignerDataset(x, indices=tec)
val_ds_clean = WignerDataset(x, indices=vac)

In [ ]:
from torch.utils.data import DataLoader 

train_loader_clean = DataLoader(train_ds_clean, batch_size=64, shuffle=True)
test_loader_clean = DataLoader(test_ds_clean, batch_size=64, shuffle=False)
val_loader_clean = DataLoader(val_ds_clean, batch_size=64, shuffle=False)

train_loader_noisy = DataLoader(train_ds_noisy, batch_size=64, shuffle=True)
test_loader_noisy = DataLoader(test_ds_noisy, batch_size=64, shuffle=False)
val_loader_noisy = DataLoader(val_ds_noisy, batch_size=64, shuffle=False)


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
from training import CNN_training
from classifier import StateClassifierCNN

model = StateClassifierCNN()

In [ ]:
# ---- CLEAN ----
CNN_training(model, train_loader_clean, val_loader_clean, device, name="clean", n_epochs=60)

In [ ]:
model = StateClassifierCNN()

In [ ]:
# ---- NOISY ----
CNN_training(model, train_loader_noisy, val_loader_noisy, device, name="noisy", n_epochs=60)

In [ ]:
from evaluation import evaluate
import pathlib

model_c = StateClassifierCNN().to(device)
model_n = StateClassifierCNN().to(device)

home = project_dir
model.load_state_dict(torch.load(f"{home}/models/noisy.pt", map_location=device, weights_only=True))
model.load_state_dict(torch.load(f"{home}/models/clean.pt", map_location=device, weights_only=True))

In [ ]:
def plot_cm(cm):
    import matplotlib.pyplot as plt
    import numpy as np

    # names = lista 7 nazw w tej samej kolejności co LABEL_TO_ID
    names = ["fock", "coherent", "vacuum", "thermal", "cat", "gkp", "binomial"]

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax.set(
        xticks=np.arange(len(names)),
        yticks=np.arange(len(names)),
        xticklabels=names,
        yticklabels=names,
        ylabel="Prawdziwa klasa",
        xlabel="Predykcja",
        title="Confusion matrix (test)",
    )
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    # liczby w kratkach
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, int(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=10,
            )

    fig.tight_layout()
    fig.savefig("confusion_noisy.png", dpi=150, bbox_inches="tight")
    plt.show()